### Extracting data from a REST API using the requests library

In [3]:
import requests
import pandas as pd

url = "https://restcountries.com/v3.1/all?fields=name,capital,population,region"

response = requests.get(url)
data = response.json()

# Parse into a clean DataFrame
countries = []
for c in data:
    countries.append({
        "name": c["name"]["common"],
        "capital": c.get("capital", ["N/A"])[0] if c.get("capital") else "N/A",
        "population": c.get("population", 0),
        "region": c.get("region", "N/A")
       
    })

df = pd.DataFrame(countries).sort_values("population", ascending=False)
df.head()

,name,capital,population,region
109,India,New Delhi,1417492000,Asia
227,China,Beijing,1408280000,Asia
148,United States,"Washington, D.C.",340110988,Americas
61,Indonesia,Jakarta,284438782,Asia
138,Pakistan,Islamabad,241499431,Asia


In [14]:
df.to_csv("countries.csv", index=False)

## Reading CSV

In [15]:
df = pd.read_csv("countries.csv")

In [16]:
df.head()

,name,capital,population,region
0,India,New Delhi,1417492000,Asia
1,China,Beijing,1408280000,Asia
2,United States,"Washington, D.C.",340110988,Americas
3,Indonesia,Jakarta,284438782,Asia
4,Pakistan,Islamabad,241499431,Asia


### Exploring The Dataset

In [18]:
df.shape

(250, 4)

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        250 non-null    object
 1   capital     246 non-null    object
 2   population  250 non-null    int64 
 3   region      250 non-null    object
dtypes: int64(1), object(3)
memory usage: 7.9+ KB


In [22]:
df.size

1000

In [24]:
df.describe()

,population
count,2.500000e+02
mean,3.207798e+07
std,1.319655e+08
min,0.000000e+00
25%,2.233542e+05
50%,5.279123e+06
75%,2.037166e+07
max,1.417492e+09


# Cleaning And Transforming The Dataset

In [27]:
df["capital"] = df["capital"].fillna("N/A")
df["region"] = df["region"].fillna("N/A")
df["population"] = df["population"].fillna(0)

#  Feature Engineering

In [34]:
df["population_category"] = df["population"].apply(
    lambda x: "High" if x > 100_000_000 else "Medium" if x > 10_000_000 else "Low")

In [35]:
df["population_millions"] = df["population"] / 1_000_000

In [37]:
df["has_capital"] = df["capital"].apply(lambda x: "Yes" if x != "N/A" else "No")

In [39]:
region_map = {
    "Asia": "AS",
    "Europe": "EU",
    "Americas": "AM",
    "Africa": "AF",
    "Oceania": "OC"}
df["region_code"] = df["region"].map(region_map).fillna("NA")

# Dropping Duplicates

In [40]:
df = df.drop_duplicates()

In [41]:
df.head()

,name,capital,population,region,population_category,population_millions,has_capital,region_code
0,India,New Delhi,1417492000,Asia,High,1417.492000,Yes,AS
1,China,Beijing,1408280000,Asia,High,1408.280000,Yes,AS
2,United States,"Washington, D.C.",340110988,Americas,High,340.110988,Yes,AM
3,Indonesia,Jakarta,284438782,Asia,High,284.438782,Yes,AS
4,Pakistan,Islamabad,241499431,Asia,High,241.499431,Yes,AS


# Sorting

In [42]:
df = df.sort_values("population",ascending=False)

In [52]:
df = df.reset_index(drop=True)

In [53]:
df.to_csv("countries_clean.csv", index=False)

#  Business Questions

In [32]:
df.groupby("region")["population"].sum().sort_values(ascending=False)

region
Asia         4724731966
Africa       1462464411
Americas     1042579783
Europe        741657922
Oceania        48059678
Antarctic          1700
Name: population, dtype: int64

In [45]:
df[["name", "population"]].head(10)

,name,population
0,India,1417492000
1,China,1408280000
2,United States,340110988
3,Indonesia,284438782
4,Pakistan,241499431
5,Nigeria,223800000
6,Brazil,213421037
7,Bangladesh,169828911
8,Russia,146028325
9,Mexico,130575786


In [47]:
df["population_category"].value_counts()

population_category
Low       156
Medium     78
High       16
Name: count, dtype: int64

In [50]:
df[df["has_capital"] == "No"]

,name,capital,population,region,population_category,population_millions,has_capital,region_code
167,Macau,N/A,685900,Asia,Low,0.6859,No,AS
240,Antarctica,N/A,1300,Antarctic,Low,0.0013,No,NA
246,Heard Island and McDonald Islands,N/A,0,Antarctic,Low,0.0000,No,NA
248,Bouvet Island,N/A,0,Antarctic,Low,0.0000,No,NA
